# Shared face cropping and half-face masking

Self-contained notebook. Requires Python 3.10+, pandas and Pillow; run in Jupyter. If needed, install with `%pip install "pandas>=2.0,<3.0" "Pillow>=9.1"` in a separate cell.

Put this notebook beside `expw_subset_metadata_comp90051.csv`. Source JPEGs default to `images/origin`. Adjust paths below, keep `RUN_FULL=False`, then use **Run → Run All Cells**.

The code verifies the team's canonical metadata SHA256, crops faces and saves `clean`, `upper_mask` and `lower_mask` PNGs, an output manifest and run settings. Default preview: 21 samples × 3 versions = 63 condition PNGs. A full run still displays only a small preview.

**Native crop size, RGB and grey 128 are provisional.** Blur/noise, PCA, models and CV are not included. Agree final image settings before full experiments. Original metadata order is preserved; failed samples stop the run. The hash checks metadata, not original-image pixels.

Before uploading a copy you have run, clear all cell outputs and save. This supplied file already has empty outputs.


In [ ]:
from pathlib import Path

RUN_FULL = False  # False: 3 samples per class; True: all 14,935 samples.

# Put this notebook beside your metadata CSV, or one folder below it.
PROJECT_DIR = Path.cwd()
CSV_NAME = "expw_subset_metadata_comp90051.csv"
if not (PROJECT_DIR / CSV_NAME).is_file() and (PROJECT_DIR.parent / CSV_NAME).is_file():
    PROJECT_DIR = PROJECT_DIR.parent

# Change PROJECT_DIR above or these two paths if your dataset is elsewhere.
METADATA_CSV = PROJECT_DIR / CSV_NAME
IMAGE_ROOT = PROJECT_DIR / "images" / "origin"
OUTPUT_ROOT = PROJECT_DIR / "shared_preprocessing_outputs"

# Provisional settings: agree the final protocol with the team.
OUTPUT_SIZE = None  # Native crop size; optionally (224, 224), for example.
COLOR_MODE = "RGB"  # "L" for grayscale.
MASK_FILL = 128
PREVIEW_PER_CLASS = 3

print("Project directory:", PROJECT_DIR)
if not METADATA_CSV.is_file():
    raise FileNotFoundError(f"Set PROJECT_DIR to the folder containing {CSV_NAME}.")
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(f"Set IMAGE_ROOT to the folder containing the JPEGs: {IMAGE_ROOT}")


In [ ]:
"""Shared ExpW cropping/masking baseline; experiment settings are provisional.

This notebook reads an existing metadata CSV. It never selects a new subset,
changes metadata, creates CV splits, or trains a model. Image settings are provisional.
"""

from __future__ import annotations

import hashlib
import json
import math
import platform
import tempfile
from pathlib import Path

import pandas as pd
import PIL
from PIL import Image, ImageDraw


VERSION = "0.1.1-single"
REFERENCE_SHA256 = (
    "deb6fa9592d87248968d7c5b67f1867ad5b589c8f2dfe3aa3ec450917461dfec"
)
EXPECTED_COUNT = 14935
CONDITIONS = ("clean", "upper_mask", "lower_mask")
REQUIRED_COLUMNS = (
    "sample_face_id", "original_image_name", "face_id", "top", "left",
    "right", "bottom", "correct_facial_class",
)


def canonical_sha256(table: pd.DataFrame) -> str:
    """Exactly the read_csv -> to_csv(LF) procedure in Hinata's notebook.

Do not sort rows/columns or normalize dtypes before calling this function.
This hashes metadata, not source-image bytes.
"""
    text = table.to_csv(index=False, lineterminator="\n")
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def verify_metadata(csv_path, *, expected_hash=REFERENCE_SHA256,
                    expected_count=EXPECTED_COUNT):
    """Return (unchanged DataFrame, canonical hash); stop on any mismatch."""
    csv_path = Path(csv_path).expanduser().resolve()
    if not csv_path.is_file():
        raise FileNotFoundError(f"Metadata CSV not found: {csv_path}")
    if (not isinstance(expected_hash, str) or len(expected_hash) != 64
            or any(c not in "0123456789abcdef" for c in expected_hash)):
        raise ValueError("expected_hash must be a 64-character lowercase SHA256.")
    if type(expected_count) is not int or expected_count < 1:
        raise ValueError("expected_count must be a positive integer.")

    # Keep the same dtype inference as the team's canonical-hash notebook.
    table = pd.read_csv(csv_path)
    actual_hash = canonical_sha256(table)
    if len(table) != expected_count or actual_hash != expected_hash:
        raise ValueError(
            "Metadata verification failed. Do not reorder/delete rows or change "
            "the reference just to bypass this check.\n"
            f"Rows: {len(table)}; expected: {expected_count}\n"
            f"Canonical SHA256: {actual_hash}\nReference: {expected_hash}"
        )
    missing = sorted(set(REQUIRED_COLUMNS) - set(table.columns))
    if missing:
        raise ValueError(f"Missing metadata columns: {missing}")
    if table[list(REQUIRED_COLUMNS)].isna().any().any():
        raise ValueError("Required metadata fields contain missing values.")
    for column in ("sample_face_id", "original_image_name"):
        if not table[column].is_unique:
            raise ValueError(f"Duplicate {column}; expected one face per source image.")
        if not table[column].map(lambda x: isinstance(x, str) and bool(x)).all():
            raise ValueError(f"{column} must contain nonempty strings.")
    if not table["sample_face_id"].str.fullmatch(r"[A-Za-z0-9_-]+").all():
        raise ValueError("sample_face_id contains unsafe filename characters.")
    for column in ("top", "left", "right", "bottom", "correct_facial_class"):
        values = pd.to_numeric(table[column], errors="raise")
        if not values.map(lambda x: math.isfinite(x) and x == int(x)).all():
            raise ValueError(f"{column} must contain finite integers.")
    if not table["correct_facial_class"].isin(range(7)).all():
        raise ValueError("Emotion labels must be integers 0 through 6.")
    return table, actual_hash


def _validate_settings(full, per_class, output_size, color_mode, mask_fill):
    if type(full) is not bool:
        raise ValueError("full must be True or False.")
    if type(per_class) is not int or per_class < 1:
        raise ValueError("per_class must be a positive integer.")
    if output_size is not None:
        if (not isinstance(output_size, (tuple, list)) or len(output_size) != 2
                or any(type(x) is not int or x < 1 for x in output_size)):
            raise ValueError("output_size must be None or (width, height) in pixels.")
    if color_mode not in ("RGB", "L"):
        raise ValueError("color_mode must be 'RGB' (colour) or 'L' (grayscale).")
    if type(mask_fill) is not int or not 0 <= mask_fill <= 255:
        raise ValueError("mask_fill must be an integer from 0 to 255.")


def crop_face(source, row, *, output_size=None, color_mode="RGB"):
    """Crop in annotation pixel coordinates; deliberately do not EXIF-rotate."""
    box = tuple(int(row[k]) for k in ("left", "top", "right", "bottom"))
    left, top, right, bottom = box
    with Image.open(source) as original:
        width, height = original.size
        if not (0 <= left < right <= width and 0 <= top < bottom <= height):
            raise ValueError(f"Invalid bbox {box} for source size {(width, height)}.")
        if right - left < 100 or bottom - top < 100:
            raise ValueError("Original face crop is smaller than 100 x 100 pixels.")
        # Coordinates already clipped by the shared data-preparation notebook.
        face = original.convert("RGB").crop(box)
    if color_mode == "L":
        face = face.convert("L")
    if output_size is not None:
        face = face.resize(tuple(output_size), Image.Resampling.BILINEAR)
    return face


def make_variants(face: Image.Image, *, mask_fill=128):
    """Each variant starts from the same clean face; no stacked corruptions."""
    if face.mode not in ("RGB", "L"):
        raise ValueError("Face image must have RGB or L mode.")
    if type(mask_fill) is not int or not 0 <= mask_fill <= 255:
        raise ValueError("mask_fill must be an integer from 0 to 255.")
    width, height = face.size
    split = height // 2
    fill = (mask_fill,) * 3 if face.mode == "RGB" else mask_fill
    upper = face.copy()
    lower = face.copy()
    upper.paste(fill, (0, 0, width, split))
    lower.paste(fill, (0, split, width, height))
    return {"clean": face.copy(), "upper_mask": upper, "lower_mask": lower}


def save_contact_sheet(manifest, output_dir):
    """Show at most three samples per class, even after a full processing run."""
    output_dir = Path(output_dir)
    examples = manifest.groupby("correct_facial_class", sort=False).head(3)
    tile_w, tile_h = 210, 220
    canvas = Image.new("RGB", (tile_w * len(CONDITIONS), tile_h * len(examples)), "white")
    draw = ImageDraw.Draw(canvas)
    for grid_row, (_, row) in enumerate(examples.iterrows()):
        for grid_col, condition in enumerate(CONDITIONS):
            x, y = grid_col * tile_w, grid_row * tile_h
            with Image.open(output_dir / row[f"{condition}_path"]) as saved:
                thumbnail = saved.convert("RGB")
            thumbnail.thumbnail((190, 170))
            canvas.paste(thumbnail, (x + (tile_w - thumbnail.width) // 2, y + 24))
            draw.text((x + 5, y + 4), condition, fill="black")
            draw.text((x + 5, y + 201),
                      f"{row['sample_face_id']} / class {row['correct_facial_class']}",
                      fill="black")
    path = output_dir / "preview_contact_sheet.png"
    canvas.save(path, format="PNG")
    return path


def _write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False) + "\n",
                         encoding="utf-8")
    temporary.replace(path)


def run_preprocessing(csv_path, image_root, output_root, *, full=False,
                      per_class=3, output_size=None, color_mode="RGB", mask_fill=128,
                      expected_hash=REFERENCE_SHA256, expected_count=EXPECTED_COUNT):
    """Verify the shared metadata, then process a preview or the full subset.

Outputs go to a new run subdirectory; input files and previous runs are untouched.
Defaults are provisional. Confirm the experiment protocol with the team before
using a full run for final results. This function does not create CV splits.
"""
    _validate_settings(full, per_class, output_size, color_mode, mask_fill)
    table, metadata_hash = verify_metadata(
        csv_path, expected_hash=expected_hash, expected_count=expected_count
    )
    image_root = Path(image_root).expanduser().resolve()
    output_root = Path(output_root).expanduser().resolve()
    if not image_root.is_dir():
        raise FileNotFoundError(f"Image directory not found: {image_root}")
    if output_root.is_relative_to(image_root):
        raise ValueError("Put output_root outside the original-image directory.")

    # groupby.head keeps original row order and original integer row indices.
    work = table if full else table.groupby("correct_facial_class", sort=False).head(per_class)
    mode = "full" if full else "preview"
    settings = {
        "code_version": VERSION,
        "protocol_status": "provisional; confirm settings with the team",
        "mode": mode,
        "preview_per_class": None if full else per_class,
        "conditions": list(CONDITIONS),
        "output_size_width_height": list(output_size) if output_size else None,
        "color_mode": color_mode,
        "mask_fill_0_to_255": mask_fill,
        "mask_boundary": "height // 2; upper [0:boundary], lower [boundary:height]",
        "operation_order": ["bbox crop", "colour conversion", "optional resize", "independent masks"],
        "resize_filter": "Pillow BILINEAR" if output_size else None,
        "exif_rotation": False,
        "output_pixels": "uint8, 0..255; model normalization is separate",
        "source_bbox_minimum_width_height": [100, 100],
        "metadata_canonical_sha256": metadata_hash,
        "hash_scope": "pandas-normalized metadata CSV including row/column order; not image bytes",
        "input_rows": len(table),
        "requested_rows": len(work),
        "metadata_sampling_seed": 90051,
        "preprocessing_randomness": "none",
        "versions": {"python": platform.python_version(), "pandas": pd.__version__,
                     "pillow": PIL.__version__},
    }
    config_hash = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()
    settings["config_sha256"] = config_hash
    output_root.mkdir(parents=True, exist_ok=True)
    run_dir = Path(tempfile.mkdtemp(prefix=f"{mode}_{config_hash[:8]}_", dir=output_root))
    _write_json(run_dir / "RUN_CONFIG.json", settings)
    for condition in CONDITIONS:
        (run_dir / condition).mkdir()

    records = []
    current_sample = None
    try:
        for done, (metadata_row, row) in enumerate(work.iterrows(), start=1):
            current_sample = row["sample_face_id"]
            source = (image_root / row["original_image_name"]).resolve()
            if not source.is_relative_to(image_root):
                raise ValueError(f"Source path for {current_sample} leaves image_root.")
            try:
                face = crop_face(source, row, output_size=output_size, color_mode=color_mode)
                variants = make_variants(face, mask_fill=mask_fill)
                record = {"metadata_row": int(metadata_row), **row.to_dict()}
                record["output_width"], record["output_height"] = face.size
                for condition, picture in variants.items():
                    relative = Path(condition) / f"{current_sample}.png"
                    picture.save(run_dir / relative, format="PNG")
                    record[f"{condition}_path"] = relative.as_posix()
                records.append(record)
            except Exception as exc:
                raise RuntimeError(
                    f"Stopped at metadata row {metadata_row}, sample {current_sample}, "
                    f"source {source}. Fix the cause; do not delete the metadata row. "
                    f"Cause: {exc}"
                ) from exc
            if done % 500 == 0 or done == len(work):
                print(f"Processed {done}/{len(work)} samples")

        manifest = pd.DataFrame(records)
        manifest.to_csv(run_dir / "preprocessed_manifest.csv", index=False, lineterminator="\n")
        contact_sheet = save_contact_sheet(manifest, run_dir)
        run_info = {
            **settings, "status": "complete", "processed_rows": len(manifest),
            "condition_png_count": len(manifest) * len(CONDITIONS),
            "class_counts": {str(k): int(v) for k, v in
                             manifest["correct_facial_class"].value_counts().sort_index().items()},
        }
        # This is the last write: a partial/failed run never gets this marker.
        _write_json(run_dir / "RUN_COMPLETE.json", run_info)
    except Exception as exc:
        _write_json(run_dir / "RUN_FAILED.json", {
            "status": "failed", "last_sample_id": current_sample,
            "fully_processed_rows": len(records), "error": str(exc),
        })
        raise

    print(f"Metadata verification passed: {metadata_hash}")
    print(f"Output: {run_dir}")
    return {"output_dir": run_dir, "manifest": manifest,
            "run_info": run_info, "contact_sheet": contact_sheet}


In [ ]:
result = run_preprocessing(
    METADATA_CSV,
    IMAGE_ROOT,
    OUTPUT_ROOT,
    full=RUN_FULL,
    per_class=PREVIEW_PER_CLASS,
    output_size=OUTPUT_SIZE,
    color_mode=COLOR_MODE,
    mask_fill=MASK_FILL,
)
print("Completed samples:", result["run_info"]["processed_rows"])
print("Condition PNGs:", result["run_info"]["condition_png_count"])
print("Output folder:", result["output_dir"])


In [ ]:
from IPython.display import Image as DisplayImage, display

display(DisplayImage(filename=str(result["contact_sheet"])))
